# Real Estate Investment Advisor
## Data Cleaning & Exploratory Data Analysis — v50

**Dataset:** `data/india_housing_prices.csv`  
**Purpose:** validate, clean, engineer, and explore the supplied property listings.

> **Scope note:** This dataset has no verified resale prices, investment outcomes, listing dates, crime rates, or infrastructure scores. Any investment label or future price created here is an assumption-based proxy, not observed performance. Analyses of amenities, parking, facing, and transport are descriptive associations only.

**v50 update:** completes the remaining brief-driven EDA analyses, adds state/city averages, expands summary exports, and centralizes chart export at the end of the notebook.


## Contents

1. Imports and reproducible setup
2. Load and validate data
3. Data quality review and cleaning
4. Feature engineering
5. Exploratory data analysis (sections 4.1–4.8)
   - Numeric distributions and IQR outlier diagnostics
   - City and state price comparisons (mean and median)
   - Area, property type, BHK, schools and hospitals
   - Security coverage and numeric relationships
   - Locality age, city BHK mix and expensive-locality ranking
   - Facing, parking, amenities and public transport
6. Executive-report visualizations
7. Data-driven EDA summary
8. Findings, limitations and next steps
9. Export cleaned data, analysis summaries and charts

Run all cells in order. Outputs are saved under `data/processed/` and `reports/`; registered charts are exported together in the final cell.


### Setup A - Imports


In [ ]:
# All imports required by this notebook are collected at the start.
from pathlib import Path
import warnings
import re

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import seaborn as sns
from IPython.display import display, Markdown
from plotly.subplots import make_subplots
from scipy import stats

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda value: f"{value:,.2f}")
sns.set_theme(style="whitegrid", context="notebook")

# Registry for charts exported together in the final notebook cell.
EXECUTIVE_FIGURES = []


### Setup B - Project paths


In [ ]:
# Find the project root whether run from the root or notebooks/ directory.
PROJECT_ROOT = next(
    (p for p in [Path.cwd(), Path.cwd().parent]
     if (p / "data" / "india_housing_prices.csv").exists()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Run from project root or notebooks/; dataset not found.")

DATA_PATH = PROJECT_ROOT / "data" / "india_housing_prices.csv"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
REPORT_DIR = PROJECT_ROOT / "reports"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)


### Setup C - Analysis assumptions


In [ ]:
# Assumptions for illustrative targets only; replace with validated outcomes.
REFERENCE_YEAR = 2026
ANNUAL_APPRECIATION_ASSUMPTION = 0.06
AGE_DEPRECIATION_PER_YEAR = 0.002
GOOD_INVESTMENT_THRESHOLD = 0.04
FORECAST_YEARS = 5
RANDOM_STATE = 42

print(f"Project root: {PROJECT_ROOT}")
print(f"Dataset: {DATA_PATH}")


## 1. Load and validate the source data

Required-column validation makes schema changes visible before analysis.


In [ ]:
EXPECTED_COLUMNS = ["ID", "State", "City", "Locality", "Property_Type", "BHK", "Size_in_SqFt", "Price_in_Lakhs", "Price_per_SqFt", "Year_Built", "Furnished_Status", "Floor_No", "Total_Floors", "Age_of_Property", "Nearby_Schools", "Nearby_Hospitals", "Public_Transport_Accessibility", "Parking_Space", "Security", "Amenities", "Facing", "Owner_Type", "Availability_Status"]
df_raw = pd.read_csv(DATA_PATH, low_memory=False)
missing_columns = sorted(set(EXPECTED_COLUMNS) - set(df_raw.columns))
if missing_columns: raise ValueError(f"Missing required columns: {missing_columns}")
duplicate_mask = df_raw.duplicated()
exact_duplicates = int(duplicate_mask.sum())
duplicate_ids = int(df_raw["ID"].duplicated().sum())
print(f"Schema valid: {len(df_raw):,} rows x {df_raw.shape[1]} columns.")
print(f"Exact duplicate rows: {exact_duplicates:,} | Duplicate IDs: {duplicate_ids:,}")


### Preview table



In [ ]:
display(df_raw.head(10))


### Column profile table



In [ ]:
schema = pd.DataFrame({
    "column": df_raw.columns,
    "dtype": df_raw.dtypes.astype(str).values,
    "non_null": df_raw.notna().sum().values,
    "missing": df_raw.isna().sum().values,
    "missing_pct": df_raw.isna().mean().values * 100,
    "unique_values": df_raw.nunique(dropna=True).values,
})
display(schema.sort_values("missing_pct", ascending=False).reset_index(drop=True))


## 2. Data quality review and cleaning

**Decisions:** remove exact duplicates; trim text and standardize missing tokens; coerce expected numeric columns; exclude rows with missing or non-positive area/price from price analysis. Other missing values are retained for review. Imputation for modeling should be fitted on training data only.

Price per square foot is recomputed as price in lakhs multiplied by 100,000 and divided by area in square feet. Assuming the source price is in lakhs of INR, the result is INR per square foot.


### Cleaning transformations



In [ ]:
df = df_raw.loc[~duplicate_mask].copy()
text_columns = df.select_dtypes(include=["object", "string"]).columns
missing_tokens = {"", "na", "n/a", "null", "none", "unknown"}
for column in text_columns:
    values = df[column].astype("string").str.strip()
    df[column] = values.mask(values.str.lower().isin(missing_tokens))
# Source price-per-sq-ft and age are recomputed below, so coerce only inputs.
numeric_columns = ["BHK", "Size_in_SqFt", "Price_in_Lakhs", "Year_Built", "Floor_No", "Total_Floors", "Nearby_Schools", "Nearby_Hospitals"]
df[numeric_columns] = df[numeric_columns].apply(pd.to_numeric, errors="coerce")
rows_before_filter = len(df)
valid_price_area = df["Size_in_SqFt"].gt(0) & df["Price_in_Lakhs"].gt(0)
df = df.loc[valid_price_area].dropna(subset=["Size_in_SqFt", "Price_in_Lakhs"]).copy()
df["Price_per_SqFt"] = df["Price_in_Lakhs"] * 100_000 / df["Size_in_SqFt"]


### Cleaning summary table



In [ ]:
quality_summary = pd.DataFrame({
    "measure": ["source rows", "exact duplicates removed", "rows removed for invalid price/area", "clean rows"],
    "count": [len(df_raw), exact_duplicates, rows_before_filter - len(df), len(df)],
})
display(quality_summary)


### Missing values table



In [ ]:
missing_pct_all = df.isna().mean().mul(100)
missingness = missing_pct_all[missing_pct_all.gt(0)].sort_values(ascending=False)
display(missingness.rename("missing_pct").to_frame() if not missingness.empty else "No missing values remain.")


### Completeness by field after cleaning
 
 Each horizontal bar shows the share of cleaned listings with a value (teal) and without a value (coral). The chart is shown even when no missing values are found, so complete fields appear as 100% filled bars.


In [ ]:
completeness_pct = 100 - missing_pct_all
field_order = missing_pct_all.sort_values().index
complete_values, missing_values = completeness_pct.loc[field_order], missing_pct_all.loc[field_order]
no_missing_values = bool(missing_values.eq(0).all())
title_note = "No missing values detected" if no_missing_values else "Complete and missing shares by field"
y = np.arange(len(field_order))
fig, ax = plt.subplots(figsize=(11, max(6, 0.34 * len(field_order))))
ax.barh(y, complete_values, color="#168C78", edgecolor="white", height=0.72, label="Value present")
ax.barh(y, missing_values, left=complete_values, color="#E76F51", edgecolor="white", height=0.72, label="Value missing")
ax.set_yticks(y)
ax.set_yticklabels(field_order)
ax.set_xlim(0, 125)
ax.set_xticks(np.arange(0, 101, 20))
ax.set_xlabel("Share of cleaned listings (%)")
ax.set_ylabel("")
ax.set_title(f"Field completeness after cleaning - {title_note}", loc="left", pad=14, weight="bold")
for row, (complete, missing) in enumerate(zip(complete_values, missing_values)):
    if complete >= 25:
        ax.text(complete - 1, row, f"{complete:.1f}% complete", ha="right", va="center", color="white", fontsize=8, weight="bold")
    else:
        ax.text(complete + 1, row, f"{complete:.1f}% complete", ha="left", va="center", color="#263238", fontsize=8)
    if missing > 0:
        ax.text(101, row, f"{missing:.1f}% missing", ha="left", va="center", color="#9C3D2E", fontsize=8, weight="bold")
ax.grid(axis="x", alpha=0.2)
ax.set_axisbelow(True)
ax.spines[["top", "right", "left"]].set_visible(False)
ax.legend(loc="lower right", frameon=False, ncol=2)
plt.tight_layout()
plt.show()


## 3. Feature engineering

School and hospital scores below are counts per 1,000 sq ft of property area, not geographic neighborhood density. The proxy targets are optional workflow examples, **not observed outcomes**: the classification rule is driven by assumed appreciation and property age; the forecast compounds current asking price at an assumed rate.


### Feature engineering transformations



In [ ]:
df["Age_of_Property"] = (REFERENCE_YEAR - df["Year_Built"]).clip(lower=0)
area_thousands = df["Size_in_SqFt"] / 1_000
df["School_Density_Score"] = df["Nearby_Schools"] / area_thousands
df["Hospital_Density_Score"] = df["Nearby_Hospitals"] / area_thousands
for column in ["Parking_Space", "Security"]:
    df[f"{column}_Binary"] = df[column].astype("string").str.lower().map({"yes": 1, "no": 0})
df["Annual_Appreciation_Proxy"] = ANNUAL_APPRECIATION_ASSUMPTION - df["Age_of_Property"] * AGE_DEPRECIATION_PER_YEAR
df["Good_Investment"] = (df["Annual_Appreciation_Proxy"] > GOOD_INVESTMENT_THRESHOLD).astype("Int64")
df["Price_After_5Y_Proxy"] = df["Price_in_Lakhs"] * (1 + ANNUAL_APPRECIATION_ASSUMPTION) ** FORECAST_YEARS
engineered = ["Price_per_SqFt", "Age_of_Property", "School_Density_Score", "Hospital_Density_Score", "Parking_Space_Binary", "Security_Binary", "Annual_Appreciation_Proxy", "Good_Investment", "Price_After_5Y_Proxy"]


### Engineered feature summary table



In [ ]:
display(df[engineered].describe().T)


### Proxy label distribution table



In [ ]:
display(df["Good_Investment"].value_counts(dropna=False).to_frame("count"))
print(f"Assumptions: {ANNUAL_APPRECIATION_ASSUMPTION:.1%} baseline; "
      f"{AGE_DEPRECIATION_PER_YEAR:.1%} age adjustment; "
      f"{GOOD_INVESTMENT_THRESHOLD:.1%} label threshold; {FORECAST_YEARS}-year scenario.")


## 4. Exploratory data analysis

### 4.1 Numeric distributions

The 99th percentile is used only as a chart display limit for readability; it does not remove observations from the cleaned data.


#### Distribution profile and outlier summary table


In [ ]:
measure_columns = ["Size_in_SqFt", "Price_in_Lakhs", "Price_per_SqFt"]
distribution_rows = []
for column in measure_columns:
    values = df[column].dropna()
    q1, q3 = values.quantile([.25, .75])
    iqr = q3 - q1
    lower_fence = q1 - 1.5 * iqr
    upper_fence = q3 + 1.5 * iqr
    outlier_count = int(((values < lower_fence) | (values > upper_fence)).sum())
    distribution_rows.append({
        "measure": column,
        "n": len(values),
        "min": values.min(),
        "q1": q1,
        "median": values.median(),
        "mean": values.mean(),
        "q3": q3,
        "max": values.max(),
        "skewness": values.skew(),
        "lower_iqr_fence": lower_fence,
        "upper_iqr_fence": upper_fence,
        "IQR_outlier_count": outlier_count,
        "IQR_outlier_pct": 100 * outlier_count / len(values) if len(values) else np.nan,
    })
distribution_summary = pd.DataFrame(distribution_rows)
display(distribution_summary.round(2))
print("IQR flags are review indicators only; no observations are deleted or winsorized by this diagnostic.")
distribution_summary.to_csv(REPORT_DIR / "distribution_outlier_summary.csv", index=False)


#### Distribution charts - shape, spread, and skew


In [ ]:
distribution_specs = [
    ("Size_in_SqFt", "Property area (sq ft)", "#007F8B", False),
    ("Price_in_Lakhs", "Asking price (lakhs)", "#E07A27", False),
    ("Price_per_SqFt", "Price per sq ft (rupees)", "#6C4AB6", True),
]
fig, axes = plt.subplots(1, 3, figsize=(17, 5))
for ax, (column, title, color, log_axis) in zip(axes, distribution_specs):
    values = df[column].dropna()
    if log_axis:
        values = values[values > 0]
        sns.histplot(x=values, bins=45, stat="density", color=color,
                     alpha=.55, edgecolor="white", ax=ax)
        sns.kdeplot(x=values, color="#222222", linewidth=1.8, ax=ax)
        ax.set_xscale("log")
        ax.set_xlabel(f"{title} - log scale")
    else:
        sns.histplot(x=values, bins=35, stat="density", color=color,
                     alpha=.65, edgecolor="white", ax=ax)
        ax.set_xlabel(title)
    median = values.median()
    ax.axvline(median, color="#C0392B", linestyle="--", linewidth=1.8,
               label=f"Median {median:,.0f}")
    ax.set_ylabel("Density")
    ax.set_title(title)
    ax.legend(frameon=False, loc="best")
    ax.text(.04, .94,
            f"n={len(values):,}\nSkew={values.skew():.2f}\nIQR={values.quantile(.75)-values.quantile(.25):,.0f}",
            transform=ax.transAxes, va="top", fontsize=9,
            bbox={"boxstyle": "round,pad=.35", "facecolor": "white", "alpha": .9})
    ax.grid(axis="y", alpha=.2)
fig.suptitle("Univariate distributions: shape, typical value, and asymmetry", fontsize=15, y=1.03)
plt.tight_layout()
plt.show()


### 4.2 City comparison - coverage and asking-price differences

The source has no date field, so this is a cross-sectional city comparison, not a time trend. The chart reports each city median relative to the overall median. A Kruskal-Wallis test and epsilon-squared effect size quantify whether differences are large enough to matter; with a very large synthetic sample, p-values alone can exaggerate trivial differences.


#### City price summary table — mean and median

In [ ]:
city_summary = (
    df.dropna(subset=["City"])
      .groupby("City", as_index=False)
      .agg(
          listings=("ID", "count"),
          mean_price_lakhs=("Price_in_Lakhs", "mean"),
          median_price_lakhs=("Price_in_Lakhs", "median"),
          mean_price_per_sqft=("Price_per_SqFt", "mean"),
          q1_price_lakhs=("Price_in_Lakhs", lambda x: x.quantile(.25)),
          q3_price_lakhs=("Price_in_Lakhs", lambda x: x.quantile(.75)),
          median_price_per_sqft=("Price_per_SqFt", "median"),
          median_area_sqft=("Size_in_SqFt", "median"),
      )
)
city_summary["median_gap_lakhs"] = (
    city_summary["median_price_lakhs"] - df["Price_in_Lakhs"].median()
)
city_summary["median_gap_pct"] = (
    100 * city_summary["median_gap_lakhs"] / df["Price_in_Lakhs"].median()
)
display(city_summary.sort_values("median_price_lakhs", ascending=False).reset_index(drop=True))
city_summary.to_csv(REPORT_DIR / "city_price_summary.csv", index=False)


#### City median deviations and statistical comparison


In [ ]:
city_groups = [part["Price_in_Lakhs"].dropna().to_numpy()
               for _, part in df.groupby("City") if len(part) > 1]
kw_city = stats.kruskal(*city_groups)
n_city, k_city = len(df), len(city_groups)
epsilon_sq_city = max(0, (kw_city.statistic - k_city + 1) / (n_city - k_city))
print(f"Kruskal-Wallis H={kw_city.statistic:,.1f}, p={kw_city.pvalue:.3g}, "
      f"epsilon-squared={epsilon_sq_city:.5f}.")
plot_city = city_summary.sort_values("median_gap_lakhs")
colors = ["#D95F02" if value < 0 else "#1B9E77"
          for value in plot_city["median_gap_lakhs"]]
fig, ax = plt.subplots(figsize=(10, max(6, .32 * len(plot_city))))
y = np.arange(len(plot_city))
ax.barh(y, plot_city["median_gap_lakhs"], color=colors, edgecolor="white")
ax.axvline(0, color="#222222", linewidth=1.2)
ax.set_yticks(y)
ax.set_yticklabels(plot_city["City"])
ax.set(title="City median asking price relative to the overall median",
       xlabel="Difference from overall median (lakhs; zero means no gap)", ylabel="")
for yi, (_, row) in enumerate(plot_city.iterrows()):
    ax.text(row["median_gap_lakhs"], yi,
            f"  {row['median_gap_pct']:+.1f}% | n={int(row['listings']):,}",
            va="center", ha="left" if row["median_gap_lakhs"] >= 0 else "right",
            fontsize=8)
ax.grid(axis="x", alpha=.2)
ax.margins(x=.24)
plt.tight_layout()
plt.show()


#### Average asking price by city

This chart addresses the brief's average-price question directly. It ranks the 15 cities with the highest **mean asking price** among cities with at least 100 records. Means are sensitive to high-priced listings, so the median and sample size remain available in `city_price_summary.csv`.

In [ ]:
MIN_CITY_LISTINGS = 100
city_mean_plot = city_summary.loc[city_summary["listings"] >= MIN_CITY_LISTINGS].nlargest(15, "mean_price_lakhs").sort_values("mean_price_lakhs")

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(city_mean_plot["City"], city_mean_plot["mean_price_lakhs"], color="#287C8E", edgecolor="white")
for y, row in enumerate(city_mean_plot.itertuples()):
    ax.text(row.mean_price_lakhs, y, f"  ₹{row.mean_price_lakhs:,.1f} L | n={row.listings:,}", va="center", fontsize=8)
ax.set(title="Cities with the highest mean asking price", xlabel="Mean asking price (₹ lakhs)", ylabel="")
ax.grid(axis="x", alpha=.2)
ax.margins(x=.28)
plt.tight_layout()
EXECUTIVE_FIGURES.append((fig, "07_city_mean_asking_price"))
plt.show()

display(city_mean_plot[["City", "listings", "mean_price_lakhs", "median_price_lakhs", "mean_price_per_sqft"]].round(2))
city_mean_plot.to_csv(REPORT_DIR / "city_mean_asking_price_top15.csv", index=False)


### Geographic view - city price gaps and market context
 
 Each dot marks an approximate city center; the source has no property coordinates. The map uses a diverging color scale centered on the overall median, so teal indicates a city median above the overall median and coral indicates below. The panel ranks the five highest and lowest city medians. Hover over a marker for listing count and price-per-square-foot details.
 
 **Interpretation note:** the dots are intentionally a consistent size because listing counts are similar across cities in this synthetic dataset. The map is for geographic context, not neighborhood-level analysis.


In [ ]:
# Approximate city-center coordinates (latitude, longitude); no parcel coordinates exist in the source.
city_coordinates = {
    "Ahmedabad": (23.0225, 72.5714), "Amritsar": (31.6340, 74.8723),
    "Bangalore": (12.9716, 77.5946), "Bhopal": (23.2599, 77.4126),
    "Bhubaneswar": (20.2961, 85.8245), "Bilaspur": (22.0797, 82.1409),
    "Chennai": (13.0827, 80.2707), "Coimbatore": (11.0168, 76.9558),
    "Cuttack": (20.4625, 85.8830), "Dehradun": (30.3165, 78.0322),
    "Durgapur": (23.5204, 87.3119), "Dwarka": (28.5921, 77.0460),
    "Faridabad": (28.4089, 77.3178), "Gaya": (24.7955, 84.9994),
    "Gurgaon": (28.4595, 77.0266), "Guwahati": (26.1445, 91.7362),
    "Haridwar": (29.9457, 78.1642), "Hyderabad": (17.3850, 78.4867),
    "Indore": (22.7196, 75.8577), "Jaipur": (26.9124, 75.7873),
    "Jamshedpur": (22.8046, 86.2029), "Jodhpur": (26.2389, 73.0243),
    "Kochi": (9.9312, 76.2673), "Kolkata": (22.5726, 88.3639),
    "Lucknow": (26.8467, 80.9462), "Ludhiana": (30.9010, 75.8573),
    "Mangalore": (12.9141, 74.8560), "Mumbai": (19.0760, 72.8777),
    "Mysore": (12.2958, 76.6394), "Nagpur": (21.1458, 79.0882),
    "New Delhi": (28.6139, 77.2090), "Noida": (28.5355, 77.3910),
    "Patna": (25.5941, 85.1376), "Pune": (18.5204, 73.8567),
    "Raipur": (21.2514, 81.6296), "Ranchi": (23.3441, 85.3096),
    "Silchar": (24.8333, 92.7789), "Surat": (21.1702, 72.8311),
    "Trivandrum": (8.5241, 76.9366), "Vijayawada": (16.5062, 80.6480),
    "Vishakhapatnam": (17.6868, 83.2185), "Warangal": (17.9689, 79.5941),
}

map_data = city_summary.copy()
map_data[["latitude", "longitude"]] = map_data["City"].apply(
    lambda city: pd.Series(city_coordinates.get(city, (np.nan, np.nan)))
)
map_data = map_data.dropna(subset=["latitude", "longitude"]).copy()
if map_data.empty:
    raise ValueError("No city coordinates matched the city summary.")

# Use a symmetric color range so positive and negative deviations are visually comparable.
color_limit = max(float(map_data["median_gap_pct"].abs().quantile(0.95)), 1.0)
top_bottom_cities = pd.concat([
    map_data.nsmallest(5, "median_gap_pct"),
    map_data.nlargest(5, "median_gap_pct"),
]).drop_duplicates("City").sort_values("median_gap_pct")
top_bottom_cities["gap_color"] = top_bottom_cities["median_gap_pct"].apply(
    lambda value: "#D66A5E" if value < 0 else "#168C78"
)

fig = make_subplots(
    rows=1, cols=2,
    specs=[[{"type": "geo"}, {"type": "xy"}]],
    column_widths=[0.62, 0.38],
    horizontal_spacing=0.035,
    subplot_titles=("City locations", "Largest city price gaps"),
)
fig.add_trace(
    go.Scattergeo(
        lon=map_data["longitude"],
        lat=map_data["latitude"],
        text=map_data["City"],
        mode="markers",
        marker={
            "size": 12,
            "color": map_data["median_gap_pct"],
            "colorscale": [[0, "#D66A5E"], [0.5, "#F4F1E8"], [1, "#168C78"]],
            "cmin": -color_limit,
            "cmax": color_limit,
            "cmid": 0,
            "opacity": 0.92,
            "line": {"color": "white", "width": 1.2},
            "colorbar": {
                "title": {"text": "Price gap<br>vs overall median"},
                "ticksuffix": "%",
                "x": 0.605, "len": 0.72, "thickness": 13,
                "outlinewidth": 0,
            },
        },
        customdata=map_data[["listings", "median_price_lakhs", "median_price_per_sqft", "median_gap_pct"]],
        hovertemplate=(
            "<b>%{text}</b><br>"
            "Listings: %{customdata[0]:,}<br>"
            "Median asking price: ₹%{customdata[1]:.1f} lakhs<br>"
            "Median price: ₹%{customdata[2]:,.0f} per sq ft<br>"
            "Gap vs overall median: %{customdata[3]:+.2f}%"
            "<extra></extra>"
        ),
        showlegend=False,
    ),
    row=1, col=1,
)
fig.add_trace(
    go.Bar(
        x=top_bottom_cities["median_gap_pct"],
        y=top_bottom_cities["City"],
        orientation="h",
        marker={"color": top_bottom_cities["gap_color"], "line": {"color": "white", "width": 0.8}},
        customdata=top_bottom_cities[["median_price_lakhs", "listings"]],
        text=[
            f"{gap:+.1f}%  |  ₹{price:.0f}L"
            for gap, price in zip(top_bottom_cities["median_gap_pct"], top_bottom_cities["median_price_lakhs"])
        ],
        textposition="outside",
        cliponaxis=False,
        hovertemplate=(
            "<b>%{y}</b><br>"
            "Gap vs overall median: %{x:+.2f}%<br>"
            "Median asking price: ₹%{customdata[0]:.1f} lakhs<br>"
            "Listings: %{customdata[1]:,}"
            "<extra></extra>"
        ),
    ),
    row=1, col=2,
)
fig.update_geos(
    lataxis_range=[6, 36], lonaxis_range=[67, 98],
    showcountries=True, countrycolor="#AAB5C0",
    showland=True, landcolor="#F4F7FA",
    showocean=True, oceancolor="#EAF4FA",
    showcoastlines=True, coastlinecolor="#8293A3",
    showframe=False,
    domain={"x": [0, 0.62], "y": [0.02, 0.96]},
)
fig.update_xaxes(
    title_text="Difference from overall median (%)",
    ticksuffix="%",
    zeroline=True, zerolinecolor="#536273", zerolinewidth=1.3,
    showgrid=True, gridcolor="#E7EBEF",
    row=1, col=2,
)
fig.update_yaxes(
    title_text="",
    showgrid=False,
    categoryorder="array",
    categoryarray=top_bottom_cities["City"].tolist(),
    row=1, col=2,
)
fig.update_layout(
    title={
        "text": (
            "Indian property listings by city"
            "<br><sup>Map color shows median asking-price gap from the overall median; right panel ranks the five highest and lowest city medians.</sup>"
        ),
        "x": 0.04, "xanchor": "left",
        "font": {"size": 21, "color": "#243447"},
    },
    height=680,
    paper_bgcolor="white", plot_bgcolor="white",
    font={"family": "Arial, sans-serif", "color": "#334155", "size": 12},
    margin={"l": 20, "r": 95, "t": 105, "b": 45},
    bargap=0.32,
    hoverlabel={"bgcolor": "#172B4D", "font_size": 13, "font_color": "white"},
)
fig.show()


### 4.3 Area and asking price - relationship strength

A large hexbin plot turns a uniform dataset into a solid block and hides the takeaway. The left panel samples listings and adds a fitted trend; the right groups area into equal-frequency bands and plots mean price with a 95% confidence interval. Spearman correlation and sample size are reported on the chart. A flat trend means this file contains little evidence that larger listed area is associated with higher asking price.


#### Sample relationship and area-band confidence chart


In [ ]:
rho_area_price, p_area_price = stats.spearmanr(
    df["Size_in_SqFt"], df["Price_in_Lakhs"], nan_policy="omit"
)
sample = df.sample(min(2_500, len(df)), random_state=RANDOM_STATE)
area_bands = (
    df.assign(Area_Band=pd.qcut(df["Size_in_SqFt"], q=20, duplicates="drop"))
      .groupby("Area_Band", observed=True)
      .agg(
          n=("Price_in_Lakhs", "size"),
          median_area=("Size_in_SqFt", "median"),
          mean_price=("Price_in_Lakhs", "mean"),
          std_price=("Price_in_Lakhs", "std"),
      )
      .reset_index()
)
area_bands["ci95"] = 1.96 * area_bands["std_price"] / np.sqrt(area_bands["n"])

fig, axes = plt.subplots(1, 2, figsize=(15, 5.8))
sns.regplot(
    data=sample, x="Size_in_SqFt", y="Price_in_Lakhs",
    scatter_kws={"s": 16, "alpha": .22, "color": "#1976A5"},
    line_kws={"color": "#D1495B", "linewidth": 2.5},
    ci=95, color="#D1495B", ax=axes[0],
)
axes[0].text(
    .04, .96,
    f"Spearman rho = {rho_area_price:.3f}\np = {p_area_price:.3g}\nN = {len(df):,}",
    transform=axes[0].transAxes, va="top", fontsize=10,
    bbox={"boxstyle": "round,pad=.4", "facecolor": "white", "edgecolor": "#BBBBBB"},
)
axes[0].set(
    title="Listing sample with fitted linear trend",
    xlabel="Property area (sq ft)",
    ylabel="Asking price (lakhs)",
)
axes[0].grid(alpha=.2)

axes[1].errorbar(
    area_bands["median_area"], area_bands["mean_price"],
    yerr=area_bands["ci95"], fmt="o-", color="#007F8B",
    ecolor="#E07A27", elinewidth=2, capsize=3, markersize=5,
)
axes[1].axhline(
    df["Price_in_Lakhs"].mean(), color="#555555", linestyle="--",
    linewidth=1.5, label=f"Overall mean = {df['Price_in_Lakhs'].mean():.1f} L",
)
axes[1].set(
    title="Mean asking price by equal-frequency area band",
    xlabel="Median area in band (sq ft)",
    ylabel="Mean price (lakhs; bars = 95% CI)",
)
axes[1].legend(frameon=False)
axes[1].grid(alpha=.2)
fig.suptitle("Is asking price associated with property area?", fontsize=15)
plt.tight_layout()
plt.show()


#### Area-band values table


In [ ]:
display(area_bands[["Area_Band", "n", "median_area", "mean_price", "ci95"]].round(2))


#### Area-band summary table


In [ ]:
area_summary = (
    df.assign(Area_Band=pd.qcut(df["Size_in_SqFt"], q=10, duplicates="drop"))
      .groupby("Area_Band", observed=True)
      .agg(
          listings=("Price_in_Lakhs", "size"),
          median_area_sqft=("Size_in_SqFt", "median"),
          q1_price_lakhs=("Price_in_Lakhs", lambda x: x.quantile(.25)),
          median_price_lakhs=("Price_in_Lakhs", "median"),
          q3_price_lakhs=("Price_in_Lakhs", lambda x: x.quantile(.75)),
      )
      .reset_index()
)
area_summary["median_gap_lakhs"] = (
    area_summary["median_price_lakhs"] - df["Price_in_Lakhs"].median()
)
display(area_summary.round(2))


#### Median asking-price gap across area bands


In [ ]:
fig, ax = plt.subplots(figsize=(10, 5.5))
colors = ["#D95F02" if value < 0 else "#1B9E77"
          for value in area_summary["median_gap_lakhs"]]
ax.axhline(0, color="#222222", linewidth=1.2)
ax.plot(area_summary["median_area_sqft"], area_summary["median_gap_lakhs"],
        color="#4472C4", linewidth=2, alpha=.8)
ax.scatter(area_summary["median_area_sqft"], area_summary["median_gap_lakhs"],
           c=colors, s=95, edgecolor="white", linewidth=1.2, zorder=3)
for row in area_summary.itertuples():
    ax.annotate(f"{row.median_gap_lakhs:+.1f} L", 
                (row.median_area_sqft, row.median_gap_lakhs),
                xytext=(0, 8), textcoords="offset points", ha="center", fontsize=8)
ax.set(title="Median price difference from all listings by area decile",
       xlabel="Median area in band (sq ft)",
       ylabel="Median asking-price gap (lakhs)")
ax.grid(alpha=.22)
plt.tight_layout()
plt.show()


### 4.4 Property type and bedroom segments

Compare asking price and price per square foot across property-type and BHK combinations. Cell counts are shown so sparse combinations can be identified rather than over-interpreted.


#### Segment summary table and group test


In [ ]:
segment_summary = (
    df.dropna(subset=["Property_Type", "BHK"])
      .groupby(["Property_Type", "BHK"], as_index=False)
      .agg(
          listings=("ID", "size"),
          median_area_sqft=("Size_in_SqFt", "median"),
          median_price_lakhs=("Price_in_Lakhs", "median"),
          median_price_per_sqft=("Price_per_SqFt", "median"),
      )
)
display(segment_summary.sort_values(["Property_Type", "BHK"]).round(2))

type_groups = [g["Price_in_Lakhs"].dropna().to_numpy()
               for _, g in df.groupby("Property_Type") if len(g) > 1]
bhk_groups = [g["Price_in_Lakhs"].dropna().to_numpy()
              for _, g in df.groupby("BHK") if len(g) > 1]
test_rows = []
for dimension, groups in [("Property_Type", type_groups), ("BHK", bhk_groups)]:
    result = stats.kruskal(*groups)
    n, k = sum(map(len, groups)), len(groups)
    effect = max(0, (result.statistic - k + 1) / (n - k))
    test_rows.append({
        "grouping": dimension, "groups": k, "n": n,
        "kruskal_H": result.statistic, "p_value": result.pvalue,
        "epsilon_squared": effect,
    })
segment_tests = pd.DataFrame(test_rows)
display(segment_tests.round(6))


#### Median price per sq ft by property type and BHK


In [ ]:
segment_wide = segment_summary.pivot(
    index="Property_Type", columns="BHK", values="median_price_per_sqft"
)
counts_wide = segment_summary.pivot(
    index="Property_Type", columns="BHK", values="listings"
)
annotations = segment_wide.copy().astype(object)
for row_name in segment_wide.index:
    for col_name in segment_wide.columns:
        value = segment_wide.loc[row_name, col_name]
        count = counts_wide.loc[row_name, col_name]
        annotations.loc[row_name, col_name] = (
            "" if pd.isna(value) or count < 100
            else f"{value:,.0f}\n(n={count:,.0f})"
        )
plot_values = segment_wide.where(counts_wide >= 100)
fig, ax = plt.subplots(figsize=(10, 6))
sns.heatmap(plot_values, annot=annotations, fmt="", cmap="YlGnBu",
            linewidths=.6, linecolor="white",
            cbar_kws={"label": "Median price per sq ft (rupees)"}, ax=ax)
ax.set(title="Price per sq ft by property type and BHK (cells with n < 100 hidden)",
       xlabel="Bedrooms (BHK)", ylabel="Property type")
plt.tight_layout()
plt.show()


### 4.5 Nearby schools and hospitals

Nearby amenity counts are listing attributes, not geographic accessibility or quality scores. The analysis reports rank association and groups counts for a descriptive comparison; it does not imply an amenity causes a price change.


#### Amenity-price association table


In [ ]:
amenity_results = []
for feature in ["Nearby_Schools", "Nearby_Hospitals"]:
    valid = df[[feature, "Price_in_Lakhs", "Price_per_SqFt"]].dropna()
    rho_price, p_price = stats.spearmanr(valid[feature], valid["Price_in_Lakhs"])
    rho_ppsf, p_ppsf = stats.spearmanr(valid[feature], valid["Price_per_SqFt"])
    amenity_results.append({
        "amenity_count": feature,
        "n": len(valid),
        "spearman_rho_vs_price": rho_price,
        "p_value_vs_price": p_price,
        "spearman_rho_vs_price_per_sqft": rho_ppsf,
        "p_value_vs_price_per_sqft": p_ppsf,
    })
amenity_association = pd.DataFrame(amenity_results)
display(amenity_association.round(5))


#### Asking price by amenity count


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
for ax, feature, color, label in zip(
    axes,
    ["Nearby_Schools", "Nearby_Hospitals"],
    ["#007F8B", "#E07A27"],
    ["Nearby schools", "Nearby hospitals"],
):
    grouped = (
        df.groupby(feature, dropna=True)
          .agg(n=("Price_in_Lakhs", "size"),
               median_price=("Price_in_Lakhs", "median"),
               q1=("Price_in_Lakhs", lambda x: x.quantile(.25)),
               q3=("Price_in_Lakhs", lambda x: x.quantile(.75)))
          .query("n >= 100")
          .reset_index()
    )
    ax.vlines(grouped[feature], grouped["q1"], grouped["q3"],
              color=color, alpha=.5, linewidth=3)
    ax.scatter(grouped[feature], grouped["median_price"], color=color,
               edgecolor="white", s=75, zorder=3)
    for row in grouped.itertuples():
        ax.annotate(f"n={row.n:,}", (getattr(row, feature), row.median_price),
                    xytext=(0, 7), textcoords="offset points", ha="center", fontsize=7)
    ax.set(title=f"Price by {label.lower()} count (n >= 100)",
           xlabel=label, ylabel="Asking price (lakhs; dot=median, line=Q1-Q3)")
    ax.grid(alpha=.2)
plt.tight_layout()
plt.show()


### 4.6 Security field coverage (crime rate is not supplied)

The dataset has a security listing attribute but no crime-rate measure. This is a data-coverage chart only; it does not estimate crime impact or investment quality.


#### Security category coverage table


In [ ]:
security_summary = (
    df["Security"].fillna("Missing")
      .value_counts(dropna=False)
      .rename_axis("security_value")
      .rename("listings")
      .reset_index()
)
security_summary["share"] = security_summary["listings"] / security_summary["listings"].sum()
display(security_summary)


#### Security category coverage chart


In [ ]:
plot_security = security_summary.sort_values("listings")
fig, ax = plt.subplots(figsize=(8, 4.8))
palette = sns.color_palette("Set2", n_colors=len(plot_security))
bars = ax.barh(plot_security["security_value"], plot_security["listings"],
               color=palette, edgecolor="white")
for bar, (_, row) in zip(bars, plot_security.iterrows()):
    ax.text(bar.get_width(), bar.get_y() + bar.get_height()/2,
            f"  {int(row['listings']):,} ({row['share']:.1%})",
            va="center", fontsize=9)
ax.set(title="Coverage of security information in listings",
       xlabel="Number of listings", ylabel="Recorded security value")
ax.grid(axis="x", alpha=.2)
ax.margins(x=.2)
plt.tight_layout()
plt.show()


### 4.7 Numeric feature relationships

Pearson correlation captures linear association; Spearman captures rank association. Both are descriptive. Correlation involving engineered ratios or the appreciation proxy can be mechanical by construction.


#### Correlation matrix table


In [ ]:
corr_cols = [
    "BHK", "Size_in_SqFt", "Price_in_Lakhs", "Price_per_SqFt",
    "Age_of_Property", "Nearby_Schools", "Nearby_Hospitals",
    "School_Density_Score", "Hospital_Density_Score",
    "Annual_Appreciation_Proxy",
]
corr = df[corr_cols].corr(numeric_only=True)
display(corr.round(3))
corr.to_csv(REPORT_DIR / "correlation_matrix.csv")


#### Correlation heatmap


In [ ]:
corr_labels = {
    "BHK": "Bedrooms (BHK)",
    "Size_in_SqFt": "Area",
    "Price_in_Lakhs": "Asking price",
    "Price_per_SqFt": "Price per sq ft",
    "Age_of_Property": "Property age",
    "Nearby_Schools": "Nearby schools",
    "Nearby_Hospitals": "Nearby hospitals",
    "School_Density_Score": "School count / area",
    "Hospital_Density_Score": "Hospital count / area",
    "Annual_Appreciation_Proxy": "Appreciation proxy",
}
corr_plot = corr.rename(index=corr_labels, columns=corr_labels)
mask = np.triu(np.ones_like(corr_plot, dtype=bool), k=1)
fig, ax = plt.subplots(figsize=(11, 8))
sns.heatmap(corr_plot, mask=mask, cmap="RdYlBu_r", center=0,
            vmin=-1, vmax=1, annot=True, fmt=".2f", square=True,
            linewidths=.7, linecolor="white",
            cbar_kws={"label": "Pearson correlation"}, ax=ax)
ax.set_title("Correlation among numeric listing features", pad=14)
plt.xticks(rotation=35, ha="right")
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()


In [ ]:
# The chart registry is initialized in Setup A so detailed EDA and executive charts export together.


## 4.8 Additional EDA coverage from the project brief

This section completes the remaining dataset-supported questions: locality property age, city-level BHK mix, expensive-locality price ranking, facing, parking, amenities, and public-transport accessibility. Each comparison reports sample size and uses asking-price metrics. Results are descriptive and do not establish causal effects or realized investment potential.

**Important date limitation:** the supplied listing schema contains no listing/posting date. Therefore, a true time-series price trend for the five most expensive localities cannot be computed. The analysis below provides a cross-sectional price ranking instead and explicitly exports that limitation.


### 4.8.1 Median property age by locality

City-qualified localities with at least 30 valid property-age records are included. The chart shows the 15 oldest localities by median property age; the full qualifying summary is exported for review. Localities are grouped with their city to avoid merging identical neighbourhood names across different cities.

In [ ]:
MIN_LOCALITY_LISTINGS = 30
locality_age_summary = (
    df.dropna(subset=["City", "Locality", "Age_of_Property"])
      .groupby(["City", "Locality"], as_index=False)
      .agg(
          listings=("ID", "count"),
          median_property_age=("Age_of_Property", "median"),
          mean_property_age=("Age_of_Property", "mean"),
          median_price_per_sqft=("Price_per_SqFt", "median"),
      )
)
locality_age_summary = locality_age_summary.loc[locality_age_summary["listings"] >= MIN_LOCALITY_LISTINGS].copy()
locality_age_summary["Location_Label"] = locality_age_summary["Locality"].astype(str) + ", " + locality_age_summary["City"].astype(str)
locality_age_plot = locality_age_summary.nlargest(15, "median_property_age").sort_values("median_property_age")

if not locality_age_plot.empty:
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.barh(locality_age_plot["Location_Label"], locality_age_plot["median_property_age"], color="#287C8E", edgecolor="white")
    for y, row in enumerate(locality_age_plot.itertuples()):
        ax.text(row.median_property_age, y, f"  {row.median_property_age:.1f} yrs | n={row.listings:,}", va="center", fontsize=8)
    ax.set(title="Localities with the highest median property age", xlabel="Median property age (years)", ylabel="")
    ax.grid(axis="x", alpha=.2)
    ax.margins(x=.25)
    plt.tight_layout()
    EXECUTIVE_FIGURES.append((fig, "08_locality_median_property_age"))
    plt.show()
else:
    print(f"No locality met the minimum sample size of {MIN_LOCALITY_LISTINGS:,} records.")

display(locality_age_summary.sort_values("median_property_age", ascending=False).head(30).round(2))
locality_age_summary.to_csv(REPORT_DIR / "locality_property_age_summary.csv", index=False)


### 4.8.2 BHK distribution across cities

The 12 cities with the largest number of valid BHK records are shown as 100% stacked bars. BHK values of six or more are grouped as `6+ BHK` to keep the chart readable. Percentages describe the sample composition within each city, not city-wide housing-stock composition.

In [ ]:
bhk_city = df.dropna(subset=["City", "BHK"]).copy()
bhk_city["BHK_numeric"] = pd.to_numeric(bhk_city["BHK"], errors="coerce")
bhk_city = bhk_city.dropna(subset=["BHK_numeric"])
bhk_city["BHK_band"] = bhk_city["BHK_numeric"].apply(lambda value: "6+ BHK" if value >= 6 else f"{value:g} BHK")
bhk_city_counts = (
    bhk_city.groupby(["City", "BHK_band"]).size().rename("listings").reset_index()
)
bhk_city_totals = bhk_city.groupby("City").size().rename("city_listings")
top_bhk_cities = bhk_city_totals.nlargest(12).index
bhk_city_counts = bhk_city_counts[bhk_city_counts["City"].isin(top_bhk_cities)]
bhk_city_mix = bhk_city_counts.pivot(index="City", columns="BHK_band", values="listings").fillna(0)
bhk_city_mix = bhk_city_mix.div(bhk_city_mix.sum(axis=1), axis=0).mul(100)
bhk_order = [f"{n} BHK" for n in range(1, 6)] + ["6+ BHK"]
bhk_city_mix = bhk_city_mix.reindex(columns=[c for c in bhk_order if c in bhk_city_mix.columns])
bhk_city_mix = bhk_city_mix.loc[bhk_city_mix.sum(axis=1).sort_values().index]

if not bhk_city_mix.empty:
    fig, ax = plt.subplots(figsize=(12, 7))
    left = np.zeros(len(bhk_city_mix))
    palette = sns.color_palette("crest", n_colors=max(1, len(bhk_city_mix.columns)))
    for color, column in zip(palette, bhk_city_mix.columns):
        values = bhk_city_mix[column].to_numpy()
        ax.barh(bhk_city_mix.index, values, left=left, label=column, color=color, edgecolor="white", linewidth=.4)
        left += values
    ax.set(title="BHK mix across the 12 largest cities in the dataset", xlabel="Share of city listings (%)", ylabel="", xlim=(0, 100))
    ax.legend(title="Property configuration", bbox_to_anchor=(1.02, 1), loc="upper left")
    ax.grid(axis="x", alpha=.2)
    plt.tight_layout()
    EXECUTIVE_FIGURES.append((fig, "09_city_bhk_distribution"))
    plt.show()
else:
    print("No valid City/BHK combinations were available for the BHK-by-city chart.")

bhk_city_mix.to_csv(REPORT_DIR / "city_bhk_mix_pct.csv")
bhk_city_counts.to_csv(REPORT_DIR / "city_bhk_listing_counts.csv", index=False)
display(bhk_city_mix.round(1))


### 4.8.3 Five highest-priced localities — cross-sectional ranking

The brief asks for price trends over time for the five most expensive localities. A time trend is not estimable because the supplied dataset does not contain a listing date or transaction date. As the closest valid alternative, this section ranks city-qualified localities by median asking price using at least 30 records and reports mean price alongside the median. This is not a time trend or an investment ranking.

In [ ]:
MIN_LOCALITY_PRICE_LISTINGS = 30
locality_price_summary = (
    df.dropna(subset=["City", "Locality", "Price_in_Lakhs", "Price_per_SqFt"])
      .groupby(["City", "Locality"], as_index=False)
      .agg(
          listings=("ID", "count"),
          mean_asking_price_lakhs=("Price_in_Lakhs", "mean"),
          median_asking_price_lakhs=("Price_in_Lakhs", "median"),
          median_price_per_sqft=("Price_per_SqFt", "median"),
      )
)
locality_price_summary = locality_price_summary.loc[locality_price_summary["listings"] >= MIN_LOCALITY_PRICE_LISTINGS].copy()
locality_price_summary["Location_Label"] = locality_price_summary["Locality"].astype(str) + ", " + locality_price_summary["City"].astype(str)
top5_localities = locality_price_summary.nlargest(5, "median_asking_price_lakhs").sort_values("median_asking_price_lakhs")

if not top5_localities.empty:
    fig, ax = plt.subplots(figsize=(10, 4.8))
    ax.barh(top5_localities["Location_Label"], top5_localities["median_asking_price_lakhs"], color="#D99B32", edgecolor="white")
    for y, row in enumerate(top5_localities.itertuples()):
        ax.text(row.median_asking_price_lakhs, y, f"  ₹{row.median_asking_price_lakhs:,.1f} L | n={row.listings:,}", va="center", fontsize=8)
    ax.set(title="Five highest-median-asking-price localities (cross-sectional)", xlabel="Median asking price (₹ lakhs)", ylabel="")
    ax.grid(axis="x", alpha=.2)
    ax.margins(x=.28)
    plt.tight_layout()
    EXECUTIVE_FIGURES.append((fig, "10_top_five_locality_price_ranking"))
    plt.show()
else:
    print(f"No locality met the minimum sample size of {MIN_LOCALITY_PRICE_LISTINGS:,} records.")

display(top5_localities.round(2))
locality_price_summary.to_csv(REPORT_DIR / "locality_price_summary.csv", index=False)
top5_localities.to_csv(REPORT_DIR / "top5_locality_price_ranking.csv", index=False)


### 4.8.4 Facing direction versus price per sq ft

Facing categories are normalized for whitespace and capitalization. Only directions with at least 100 records are displayed. The chart compares median price per square foot; sample counts and means are retained in the summary. This is an unadjusted association and may reflect location and property mix.

In [ ]:
facing_data = df.dropna(subset=["Facing", "Price_per_SqFt"]).copy()
facing_data["Facing_Category"] = facing_data["Facing"].astype("string").str.strip().str.replace(r"\s+", " ", regex=True).str.title()
facing_summary = (
    facing_data.groupby("Facing_Category", as_index=False)
      .agg(
          listings=("ID", "count"),
          mean_price_per_sqft=("Price_per_SqFt", "mean"),
          median_price_per_sqft=("Price_per_SqFt", "median"),
          median_asking_price_lakhs=("Price_in_Lakhs", "median"),
      )
)
facing_summary = facing_summary.loc[facing_summary["listings"] >= 100].sort_values("median_price_per_sqft")

if not facing_summary.empty:
    fig, ax = plt.subplots(figsize=(10, 5.5))
    ax.barh(facing_summary["Facing_Category"], facing_summary["median_price_per_sqft"], color="#3A9D78", edgecolor="white")
    for y, row in enumerate(facing_summary.itertuples()):
        ax.text(row.median_price_per_sqft, y, f"  ₹{row.median_price_per_sqft:,.0f} | n={row.listings:,}", va="center", fontsize=8)
    ax.set(title="Median asking price per sq ft by facing direction", xlabel="Median price per sq ft (₹)", ylabel="")
    ax.grid(axis="x", alpha=.2)
    ax.margins(x=.28)
    plt.tight_layout()
    EXECUTIVE_FIGURES.append((fig, "11_facing_price_per_sqft"))
    plt.show()
else:
    print("No facing category met the minimum sample size of 100 records.")

display(facing_summary.round(2))
facing_summary.to_csv(REPORT_DIR / "facing_price_summary.csv", index=False)


### 4.8.5 Parking availability versus asking price

The source field may be represented as a numeric parking-space count or a text Yes/No value. This cell harmonizes numeric values into 0, 1, and 2+ spaces, while retaining non-numeric categories as supplied. Categories with fewer than 100 records are excluded from the chart. The analysis is descriptive and does not control for property size, location, or type.

In [ ]:
parking_data = df.dropna(subset=["Parking_Space", "Price_in_Lakhs", "Price_per_SqFt"]).copy()
parking_raw = parking_data["Parking_Space"].astype("string").str.strip()
parking_numeric = pd.to_numeric(parking_raw, errors="coerce")
parking_label = pd.Series(index=parking_data.index, dtype="string")
numeric_mask = parking_numeric.notna()
parking_label.loc[numeric_mask & parking_numeric.le(0)] = "0 spaces"
parking_label.loc[numeric_mask & parking_numeric.eq(1)] = "1 space"
parking_label.loc[numeric_mask & parking_numeric.ge(2)] = "2+ spaces"
text_values = parking_raw.loc[~numeric_mask].str.lower()
parking_label.loc[~numeric_mask & text_values.isin(["yes", "available", "true"])] = "Parking available"
parking_label.loc[~numeric_mask & text_values.isin(["no", "none", "not available", "false"])] = "No parking"
other_mask = parking_label.isna()
parking_label.loc[other_mask] = parking_raw.loc[other_mask].str.title()
parking_data["Parking_Category"] = parking_label
parking_summary = (
    parking_data.groupby("Parking_Category", as_index=False)
      .agg(
          listings=("ID", "count"),
          mean_asking_price_lakhs=("Price_in_Lakhs", "mean"),
          median_asking_price_lakhs=("Price_in_Lakhs", "median"),
          median_price_per_sqft=("Price_per_SqFt", "median"),
      )
)
parking_summary = parking_summary.loc[parking_summary["listings"] >= 100].sort_values("median_price_per_sqft")

if not parking_summary.empty:
    fig, ax = plt.subplots(figsize=(10, 5.5))
    ax.barh(parking_summary["Parking_Category"], parking_summary["median_price_per_sqft"], color="#D99B32", edgecolor="white")
    for y, row in enumerate(parking_summary.itertuples()):
        ax.text(row.median_price_per_sqft, y, f"  ₹{row.median_price_per_sqft:,.0f} | n={row.listings:,}", va="center", fontsize=8)
    ax.set(title="Median asking price per sq ft by parking category", xlabel="Median price per sq ft (₹)", ylabel="")
    ax.grid(axis="x", alpha=.2)
    ax.margins(x=.28)
    plt.tight_layout()
    EXECUTIVE_FIGURES.append((fig, "12_parking_price_comparison"))
    plt.show()
else:
    print("No parking category met the minimum sample size of 100 records.")

display(parking_summary.round(2))
parking_summary.to_csv(REPORT_DIR / "parking_price_summary.csv", index=False)


### 4.8.6 Amenities versus price per sq ft

The `Amenities` field is treated as a list of delimited tags (comma, semicolon, pipe, or slash separated). Repeated tags within a listing count once. Blank/none/not-specified values are not treated as an amenity. The analysis compares tag-level median price per sq ft and the number of amenity tags per listing. These are unadjusted associations; they do not establish that amenities cause higher prices.

In [ ]:
AMENITY_EMPTY_TOKENS = {"", "na", "n/a", "none", "null", "unknown", "not specified", "no amenities", "nan"}
def parse_amenity_tokens(value):
    if pd.isna(value):
        return []
    raw_parts = re.split(r"[,;|/]+", str(value))
    tokens = []
    seen = set()
    for part in raw_parts:
        token = re.sub(r"\s+", " ", part).strip()
        key = token.casefold()
        if token and key not in AMENITY_EMPTY_TOKENS and key not in seen:
            tokens.append(token)
            seen.add(key)
    return tokens

amenity_data = df.copy()
amenity_data["Amenity_Tokens"] = amenity_data["Amenities"].apply(parse_amenity_tokens)
amenity_data["Amenity_Count"] = amenity_data["Amenity_Tokens"].str.len()
amenity_long = amenity_data[["ID", "Price_per_SqFt", "Price_in_Lakhs", "Amenity_Tokens"]].explode("Amenity_Tokens")
amenity_long = amenity_long.dropna(subset=["Amenity_Tokens", "Price_per_SqFt"])
amenity_summary = (
    amenity_long.groupby("Amenity_Tokens", as_index=False)
      .agg(
          listings=("ID", "nunique"),
          mean_price_per_sqft=("Price_per_SqFt", "mean"),
          median_price_per_sqft=("Price_per_SqFt", "median"),
          median_asking_price_lakhs=("Price_in_Lakhs", "median"),
      )
      .rename(columns={"Amenity_Tokens": "Amenity"})
)
amenity_summary = amenity_summary.loc[amenity_summary["listings"] >= 100].copy()
top_amenities = amenity_summary.nlargest(12, "listings").sort_values("median_price_per_sqft")

if not top_amenities.empty:
    fig, ax = plt.subplots(figsize=(11, 6))
    ax.barh(top_amenities["Amenity"], top_amenities["median_price_per_sqft"], color="#3A9D78", edgecolor="white")
    for y, row in enumerate(top_amenities.itertuples()):
        ax.text(row.median_price_per_sqft, y, f"  ₹{row.median_price_per_sqft:,.0f} | n={row.listings:,}", va="center", fontsize=8)
    ax.set(title="Median asking price per sq ft for commonly listed amenities", xlabel="Median price per sq ft (₹)", ylabel="")
    ax.grid(axis="x", alpha=.2)
    ax.margins(x=.28)
    plt.tight_layout()
    EXECUTIVE_FIGURES.append((fig, "13_amenities_price_per_sqft"))
    plt.show()
else:
    print("No amenity tag had at least 100 listings; check delimiter and field format in the raw data.")

amenity_count_summary = (
    amenity_data.groupby("Amenity_Count", as_index=False)
      .agg(
          listings=("ID", "count"),
          median_price_per_sqft=("Price_per_SqFt", "median"),
          mean_price_per_sqft=("Price_per_SqFt", "mean"),
          median_asking_price_lakhs=("Price_in_Lakhs", "median"),
      )
)
amenity_count_summary = amenity_count_summary.loc[amenity_count_summary["listings"] >= 100].sort_values("Amenity_Count")
valid_amenity_count = amenity_data.dropna(subset=["Amenity_Count", "Price_per_SqFt"])
if valid_amenity_count["Amenity_Count"].nunique() > 1:
    amenity_count_rho, amenity_count_p = stats.spearmanr(valid_amenity_count["Amenity_Count"], valid_amenity_count["Price_per_SqFt"])
else:
    amenity_count_rho, amenity_count_p = np.nan, np.nan
print(f"Spearman association: amenity-tag count vs price per sq ft = {amenity_count_rho:.3f}; p={amenity_count_p:.3g} (descriptive only).")
display(top_amenities.round(2))
display(amenity_count_summary.round(2))
amenity_summary.to_csv(REPORT_DIR / "amenity_tag_price_summary.csv", index=False)
amenity_count_summary.to_csv(REPORT_DIR / "amenity_count_price_summary.csv", index=False)


### 4.8.7 Public-transport accessibility versus price and proxy label

Accessibility categories are standardized for display, then compared on median price per sq ft. The table also reports the share carrying the project's formula-generated `Good_Investment` label. That percentage is a rule-derived proxy, not observed investment potential. A high/low category association is not causal and may reflect city or property mix.

In [ ]:
transport_data = df.dropna(subset=["Public_Transport_Accessibility", "Price_per_SqFt"]).copy()
transport_data["Transport_Category"] = transport_data["Public_Transport_Accessibility"].astype("string").str.strip().str.replace(r"\s+", " ", regex=True).str.title()
transport_summary = (
    transport_data.groupby("Transport_Category", as_index=False)
      .agg(
          listings=("ID", "count"),
          mean_price_per_sqft=("Price_per_SqFt", "mean"),
          median_price_per_sqft=("Price_per_SqFt", "median"),
          median_asking_price_lakhs=("Price_in_Lakhs", "median"),
          proxy_good_investment_pct=("Good_Investment", lambda values: 100 * pd.to_numeric(values, errors="coerce").mean()),
      )
)
transport_summary = transport_summary.loc[transport_summary["listings"] >= 100].sort_values("median_price_per_sqft")

if not transport_summary.empty:
    fig, ax = plt.subplots(figsize=(10, 5.5))
    ax.barh(transport_summary["Transport_Category"], transport_summary["median_price_per_sqft"], color="#287C8E", edgecolor="white")
    for y, row in enumerate(transport_summary.itertuples()):
        ax.text(row.median_price_per_sqft, y, f"  ₹{row.median_price_per_sqft:,.0f} | n={row.listings:,}", va="center", fontsize=8)
    ax.set(title="Median asking price per sq ft by public-transport accessibility", xlabel="Median price per sq ft (₹)", ylabel="")
    ax.grid(axis="x", alpha=.2)
    ax.margins(x=.28)
    plt.tight_layout()
    EXECUTIVE_FIGURES.append((fig, "14_transport_accessibility_price_per_sqft"))
    plt.show()
else:
    print("No transport-accessibility category met the minimum sample size of 100 records.")

display(transport_summary.round(2))
transport_summary.to_csv(REPORT_DIR / "transport_accessibility_price_summary.csv", index=False)


### 4.8.8 Coverage status and interpretation

| Brief question | v50 treatment |
|---|---|
| Median property age by locality | Added; locality groups require at least 30 records. |
| BHK distribution across cities | Added; top 12 cities, within-city percentage mix. |
| Top five expensive localities | Cross-sectional ranking added. A time trend is not possible without a date field. |
| Price per sq ft by facing | Added; groups require at least 100 records. |
| Parking versus price | Added; supports numeric parking counts and Yes/No labels. |
| Amenities versus price per sq ft | Added; parses common delimited tags and reports tag counts. |
| Public transport accessibility | Added; compares price per sq ft and separately reports the assumption-based proxy-label share. |
| Crime and infrastructure impact | Not estimable from this dataset; no verified crime-rate or infrastructure-score fields are supplied. |

All charts are descriptive. They do not adjust for confounding among city, locality, property type, area, age, or other listing attributes. The outputs should be read as dataset diagnostics and segmentation summaries, not market-wide estimates.


## 5. Executive-report visualizations

This section adds a concise set of charts designed for an executive report. The emphasis is on **market footprint, geographic price positioning, property-segment differences, and listing composition**.

These are descriptive views of the supplied listing dataset. Listing counts represent the composition of this dataset, not the full Indian housing market. Asking prices are not verified transaction prices. Groups with small samples are filtered from selected comparisons to reduce unstable summaries.

The summary tables are exported to `reports/`, and registered charts are exported together to `reports/figures/` in the final cell.

### 5.1 Executive snapshot — headline indicators

A compact KPI panel gives an executive reader the scale and central tendency of the cleaned dataset before interpreting the detailed charts.

In [ ]:
# Executive KPI snapshot: descriptive statistics from the cleaned listings.
kpi_values = [
    ("Clean listings", f"{len(df):,}"),
    ("States represented", f"{df['State'].nunique():,}"),
    ("Cities represented", f"{df['City'].nunique():,}"),
    ("Median asking price", f"₹{df['Price_in_Lakhs'].median():,.1f} L"),
    ("Median price / sq ft", f"₹{df['Price_per_SqFt'].median():,.0f}"),
    ("Median property area", f"{df['Size_in_SqFt'].median():,.0f} sq ft"),
]

fig, axes = plt.subplots(2, 3, figsize=(15, 5.8))
for ax, (label, value) in zip(axes.flat, kpi_values):
    ax.set_facecolor("#F3F6F8")
    ax.text(0.5, 0.62, value, ha="center", va="center",
            fontsize=20, fontweight="bold", color="#17324D")
    ax.text(0.5, 0.28, label, ha="center", va="center",
            fontsize=10, color="#526575")
    ax.set_xticks([])
    ax.set_yticks([])
    for spine in ax.spines.values():
        spine.set_visible(False)

fig.suptitle("Executive snapshot | cleaned property listings", fontsize=16, y=1.02)
plt.tight_layout()
EXECUTIVE_FIGURES.append((fig, "01_executive_kpi_snapshot"))
plt.show()

executive_kpis = pd.DataFrame(kpi_values, columns=["indicator", "value"])
display(executive_kpis)
executive_kpis.to_csv(REPORT_DIR / "executive_kpis.csv", index=False)


### 5.2 Geographic footprint — state listing composition and average price per sq ft

The left panel shows which states contribute the most records to this dataset. The right panel compares **mean asking price per square foot** for those same states, directly addressing the state-level average price-intensity question. A larger listing count is dataset composition, not real-world market share. Median and mean summaries are retained in the exported state table.


In [ ]:
state_summary = (
    df.dropna(subset=["State"])
      .groupby("State", as_index=False)
      .agg(
          listings=("ID", "count"),
          mean_asking_price_lakhs=("Price_in_Lakhs", "mean"),
          median_asking_price_lakhs=("Price_in_Lakhs", "median"),
          mean_price_per_sqft=("Price_per_SqFt", "mean"),
          median_price_per_sqft=("Price_per_SqFt", "median"),
          median_area_sqft=("Size_in_SqFt", "median"),
      )
)
state_summary["listing_share_pct"] = 100 * state_summary["listings"] / state_summary["listings"].sum()
state_top = state_summary.nlargest(12, "listings").sort_values("listings")

fig, axes = plt.subplots(1, 2, figsize=(15, 7))
axes[0].barh(state_top["State"], state_top["listings"], color="#287C8E", edgecolor="white")
for y, row in enumerate(state_top.itertuples()):
    axes[0].text(row.listings, y, f"  {row.listings:,} ({row.listing_share_pct:.1f}%)", va="center", fontsize=8)
axes[0].set(title="Top states by dataset listing count", xlabel="Listings in supplied dataset", ylabel="")
axes[0].grid(axis="x", alpha=.2)
axes[0].margins(x=.25)

state_price = state_top.sort_values("mean_price_per_sqft")
axes[1].barh(state_price["State"], state_price["mean_price_per_sqft"], color="#D98B45", edgecolor="white")
for y, row in enumerate(state_price.itertuples()):
    axes[1].text(row.mean_price_per_sqft, y, f"  ₹{row.mean_price_per_sqft:,.0f}/sq ft", va="center", fontsize=8)
axes[1].set(title="Mean asking price per sq ft in the same states", xlabel="Mean asking price per sq ft (₹)", ylabel="")
axes[1].grid(axis="x", alpha=.2)
axes[1].margins(x=.25)

fig.suptitle("State-level dataset footprint and average price intensity", fontsize=15, y=1.02)
plt.tight_layout()
EXECUTIVE_FIGURES.append((fig, "02_state_listing_and_price_comparison"))
plt.show()

display(state_summary.sort_values("listings", ascending=False).head(12).round(2))
state_summary.to_csv(REPORT_DIR / "state_listing_price_summary.csv", index=False)


### 5.3 City market positioning — price level versus price per sq ft

Each bubble represents a city. The horizontal axis shows median price per square foot, the vertical axis shows median asking price, and bubble size represents the number of listings in the dataset. Dashed reference lines mark the overall medians. This helps distinguish cities with higher total asking prices from cities with higher price intensity per unit area.

In [ ]:
city_plot = city_summary.dropna(
    subset=["median_price_lakhs", "median_price_per_sqft", "listings"]
).copy()

fig, ax = plt.subplots(figsize=(12, 7))
sizes = 35 + 420 * city_plot["listings"] / city_plot["listings"].max()
scatter = ax.scatter(
    city_plot["median_price_per_sqft"],
    city_plot["median_price_lakhs"],
    s=sizes,
    c=city_plot["median_gap_pct"],
    cmap="RdYlGn",
    alpha=.78,
    edgecolor="white",
    linewidth=.8,
)
ax.axvline(df["Price_per_SqFt"].median(), color="#536575",
           linestyle="--", linewidth=1.2, label="Overall median price / sq ft")
ax.axhline(df["Price_in_Lakhs"].median(), color="#9A5B32",
           linestyle=":", linewidth=1.5, label="Overall median asking price")

# Label the largest cities by listing count to avoid an unreadable label cloud.
for row in city_plot.nlargest(8, "listings").itertuples():
    ax.annotate(row.City, (row.median_price_per_sqft, row.median_price_lakhs),
                xytext=(5, 5), textcoords="offset points", fontsize=8)

cbar = fig.colorbar(scatter, ax=ax)
cbar.set_label("Median asking-price gap vs overall median (%)")
ax.set(
    title="City positioning: median asking price vs price per sq ft",
    xlabel="Median asking price per sq ft (₹)",
    ylabel="Median asking price (lakhs)",
)
ax.legend(frameon=True, loc="best")
ax.grid(alpha=.2)
plt.tight_layout()
EXECUTIVE_FIGURES.append((fig, "03_city_market_positioning"))
plt.show()


### 5.4 Property segment economics — asking-price distribution by property type

Median values alone can hide substantial variation. This boxplot compares the spread of asking prices across property types. For readability, the chart display is restricted to the 1st–99th percentile range of asking prices; the underlying cleaned dataset is not trimmed. Sample sizes are shown in the category labels.

In [ ]:
type_counts = df["Property_Type"].value_counts(dropna=True)
type_order = type_counts.index.tolist()
type_order = [name for name in type_order if type_counts[name] >= 100]
box_data = df[df["Property_Type"].isin(type_order)].copy()
display_low, display_high = box_data["Price_in_Lakhs"].quantile([.01, .99])
box_data_display = box_data[box_data["Price_in_Lakhs"].between(display_low, display_high)].copy()
type_labels = {
    name: f"{name} (n={int((df['Property_Type'] == name).sum()):,})"
    for name in type_order
}
box_data_display["Property_Type_Label"] = box_data_display["Property_Type"].map(type_labels)
ordered_labels = [type_labels[name] for name in type_order]

fig, ax = plt.subplots(figsize=(12, 6))
sns.boxplot(
    data=box_data_display,
    x="Property_Type_Label",
    y="Price_in_Lakhs",
    order=ordered_labels,
    color="#79A9A5",
    showfliers=False,
    ax=ax,
)
ax.set(
    title="Asking-price spread by property type (display limited to 1st–99th percentiles)",
    xlabel="Property type",
    ylabel="Asking price (lakhs)",
)
ax.tick_params(axis="x", rotation=20)
ax.grid(axis="y", alpha=.2)
plt.tight_layout()
EXECUTIVE_FIGURES.append((fig, "04_asking_price_by_property_type"))
plt.show()

property_type_exec = (
    df.groupby("Property_Type", dropna=True)
      .agg(
          listings=("ID", "count"),
          median_asking_price_lakhs=("Price_in_Lakhs", "median"),
          median_price_per_sqft=("Price_per_SqFt", "median"),
          median_area_sqft=("Size_in_SqFt", "median"),
      )
      .reset_index()
      .sort_values("listings", ascending=False)
)
display(property_type_exec.round(2))
property_type_exec.to_csv(REPORT_DIR / "property_type_executive_summary.csv", index=False)


### 5.5 Property age and furnishing — price-per-square-foot comparisons

This paired view compares median price per square foot across property-age bands and furnishing categories. It is descriptive, not causal: differences may also reflect city mix, property type, area, and other unobserved factors. Only groups with at least 100 listings are displayed.

In [ ]:
age_bins = [-1, 5, 10, 20, 30, np.inf]
age_labels = ["0–5 years", "6–10 years", "11–20 years", "21–30 years", "30+ years"]
age_segment = df.copy()
age_segment["Property_Age_Band"] = pd.cut(
    age_segment["Age_of_Property"], bins=age_bins, labels=age_labels
)
age_summary = (
    age_segment.dropna(subset=["Property_Age_Band"])
      .groupby("Property_Age_Band", observed=True)
      .agg(
          listings=("ID", "count"),
          median_price_per_sqft=("Price_per_SqFt", "median"),
          q1=("Price_per_SqFt", lambda x: x.quantile(.25)),
          q3=("Price_per_SqFt", lambda x: x.quantile(.75)),
      )
      .reset_index()
)
age_summary = age_summary[age_summary["listings"] >= 100]

furnishing_summary = (
    df.dropna(subset=["Furnished_Status"])
      .groupby("Furnished_Status")
      .agg(
          listings=("ID", "count"),
          median_price_per_sqft=("Price_per_SqFt", "median"),
          q1=("Price_per_SqFt", lambda x: x.quantile(.25)),
          q3=("Price_per_SqFt", lambda x: x.quantile(.75)),
      )
      .reset_index()
)
furnishing_summary = furnishing_summary[furnishing_summary["listings"] >= 100]

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
if not age_summary.empty:
    x = np.arange(len(age_summary))
    axes[0].errorbar(
        x, age_summary["median_price_per_sqft"],
        yerr=[
            age_summary["median_price_per_sqft"] - age_summary["q1"],
            age_summary["q3"] - age_summary["median_price_per_sqft"],
        ],
        fmt="o-", color="#287C8E", capsize=4, linewidth=2, markersize=6,
    )
    axes[0].set_xticks(x)
    axes[0].set_xticklabels(age_summary["Property_Age_Band"], rotation=20, ha="right")
axes[0].set(title="Price per sq ft by property age",
            xlabel="Property age band", ylabel="Median price per sq ft (₹; bars = Q1–Q3)")
axes[0].grid(axis="y", alpha=.2)

if not furnishing_summary.empty:
    furnishing_summary = furnishing_summary.sort_values("median_price_per_sqft")
    x = np.arange(len(furnishing_summary))
    axes[1].errorbar(
        x, furnishing_summary["median_price_per_sqft"],
        yerr=[
            furnishing_summary["median_price_per_sqft"] - furnishing_summary["q1"],
            furnishing_summary["q3"] - furnishing_summary["median_price_per_sqft"],
        ],
        fmt="o", color="#D98B45", capsize=4, linewidth=2, markersize=7,
    )
    axes[1].set_xticks(x)
    axes[1].set_xticklabels(furnishing_summary["Furnished_Status"], rotation=20, ha="right")
axes[1].set(title="Price per sq ft by furnishing status",
            xlabel="Furnishing status", ylabel="Median price per sq ft (₹; bars = Q1–Q3)")
axes[1].grid(axis="y", alpha=.2)

fig.suptitle("Property characteristics and asking-price intensity", fontsize=15, y=1.02)
plt.tight_layout()
EXECUTIVE_FIGURES.append((fig, "05_property_age_and_furnishing_comparison"))
plt.show()

display(age_summary.round(2))
display(furnishing_summary.round(2))
age_summary.to_csv(REPORT_DIR / "property_age_price_summary.csv", index=False)
furnishing_summary.to_csv(REPORT_DIR / "furnishing_price_summary.csv", index=False)


### 5.6 Listing composition — availability and ownership profile

This chart provides context on the records represented in the dataset. It is a profile of the supplied listings, not a measure of unsold inventory or ownership conditions across the wider market.

In [ ]:
profile_fields = ["Availability_Status", "Owner_Type", "Furnished_Status"]
profile_rows = []
for field in profile_fields:
    counts = (
        df[field].fillna("Missing")
          .value_counts(dropna=False)
          .rename_axis("category")
          .rename("listings")
          .reset_index()
    )
    counts["field"] = field
    counts["share_pct"] = 100 * counts["listings"] / counts["listings"].sum()
    profile_rows.append(counts)

listing_profile = pd.concat(profile_rows, ignore_index=True)
fig, axes = plt.subplots(1, len(profile_fields), figsize=(16, 6))
for ax, field in zip(axes, profile_fields):
    part = listing_profile[listing_profile["field"] == field].sort_values("listings")
    ax.barh(part["category"], part["listings"], color="#6C91A3", edgecolor="white")
    ax.set_title(field.replace("_", " "))
    ax.set_xlabel("Listings")
    ax.grid(axis="x", alpha=.2)
    ax.margins(x=.25)
    for y, row in enumerate(part.itertuples()):
        ax.text(row.listings, y, f"  {row.share_pct:.1f}%", va="center", fontsize=8)

fig.suptitle("Composition of listings by availability, ownership and furnishing", fontsize=15, y=1.02)
plt.tight_layout()
EXECUTIVE_FIGURES.append((fig, "06_listing_composition_profile"))
plt.show()

display(listing_profile.sort_values(["field", "listings"], ascending=[True, False]).round(2))
listing_profile.to_csv(REPORT_DIR / "listing_profile_summary.csv", index=False)


### 5.7 Executive interpretation checklist

Use the charts above to write the executive narrative after running the notebook. Report the strongest observed patterns with their sample sizes and medians. Avoid describing listing counts as market share, asking prices as transaction prices, or descriptive associations as causal effects. Where city or segment comparisons are based on uneven sample sizes, state that limitation explicitly.

In [ ]:
# Compact, data-driven talking points to support the executive report.
top_state = state_summary.sort_values("listings", ascending=False).iloc[0]
top_city_by_count = city_summary.sort_values("listings", ascending=False).iloc[0]
highest_city_median = city_summary.sort_values("median_price_lakhs", ascending=False).iloc[0]
lowest_city_median = city_summary.sort_values("median_price_lakhs", ascending=True).iloc[0]

print("EXECUTIVE REPORT TALKING POINTS")
print(f"• The cleaned dataset contains {len(df):,} listings across "
      f"{df['State'].nunique():,} states and {df['City'].nunique():,} cities.")
print(f"• {top_state['State']} contributes the largest state-level record count "
      f"({int(top_state['listings']):,}; {top_state['listing_share_pct']:.1f}% of this dataset).")
print(f"• {top_city_by_count['City']} has the largest city-level record count "
      f"({int(top_city_by_count['listings']):,}).")
print(f"• Highest city median asking price: {highest_city_median['City']} "
      f"(₹{highest_city_median['median_price_lakhs']:.1f} lakhs; "
      f"n={int(highest_city_median['listings']):,}).")
print(f"• Lowest city median asking price: {lowest_city_median['City']} "
      f"(₹{lowest_city_median['median_price_lakhs']:.1f} lakhs; "
      f"n={int(lowest_city_median['listings']):,}).")
print("• These are descriptive listing-data comparisons, not verified transaction-market estimates.")


## 6. Data-driven EDA summary

This summary converts the main tests into plain-language checks. Treat effect sizes as more useful than p-values when the dataset contains many rows.


### EDA evidence table


In [ ]:
max_city_gap_pct = city_summary["median_gap_pct"].abs().max()
area_band_median_range = (
    area_summary["median_price_lakhs"].max()
    - area_summary["median_price_lakhs"].min()
)
property_type_effect = segment_tests.loc[
    segment_tests["grouping"] == "Property_Type", "epsilon_squared"
].iloc[0]
bhk_effect = segment_tests.loc[
    segment_tests["grouping"] == "BHK", "epsilon_squared"
].iloc[0]
diagnostics = pd.DataFrame([
    {
        "question": "Does area track asking price?",
        "measure": "Spearman rho",
        "estimate": rho_area_price,
        "p_value": p_area_price,
        "context": "Near 0 means weak monotonic association.",
    },
    {
        "question": "How large are city price differences?",
        "measure": "Kruskal epsilon-squared",
        "estimate": epsilon_sq_city,
        "p_value": kw_city.pvalue,
        "context": f"Largest city median gap is {max_city_gap_pct:.2f}% from overall median.",
    },
    {
        "question": "Do property types differ in asking price?",
        "measure": "Kruskal epsilon-squared",
        "estimate": property_type_effect,
        "p_value": segment_tests.loc[segment_tests["grouping"] == "Property_Type", "p_value"].iloc[0],
        "context": "Small effect size means little practical separation.",
    },
    {
        "question": "Do BHK groups differ in asking price?",
        "measure": "Kruskal epsilon-squared",
        "estimate": bhk_effect,
        "p_value": segment_tests.loc[segment_tests["grouping"] == "BHK", "p_value"].iloc[0],
        "context": "Compare effect size with the segment counts.",
    },
    {
        "question": "Do amenity counts track price per sq ft?",
        "measure": "Spearman rho: schools / hospitals",
        "estimate": (
            amenity_association["spearman_rho_vs_price_per_sqft"].iloc[0],
            amenity_association["spearman_rho_vs_price_per_sqft"].iloc[1],
        ),
        "p_value": (
            amenity_association["p_value_vs_price_per_sqft"].iloc[0],
            amenity_association["p_value_vs_price_per_sqft"].iloc[1],
        ),
        "context": "Reported in school-count, hospital-count order.",
    },
])
display(diagnostics)


### Automatically generated interpretation


In [ ]:
print("EDA interpretation")
if abs(rho_area_price) < 0.10:
    print(f"- Area and asking price have a weak rank association (rho={rho_area_price:.3f}).")
else:
    print(f"- Area and asking price have a non-trivial rank association (rho={rho_area_price:.3f}); inspect the density plot for shape and outliers.")
if epsilon_sq_city < 0.01:
    print(f"- City has a negligible rank effect on asking price (epsilon-squared={epsilon_sq_city:.5f}); max median gap is {max_city_gap_pct:.2f}%.")
else:
    print(f"- City differences merit further review (epsilon-squared={epsilon_sq_city:.5f}); check sample mix and the city summary.")
if area_band_median_range < 0.05 * df["Price_in_Lakhs"].median():
    print(f"- Area-band medians span only {area_band_median_range:.2f} lakhs, less than 5% of the overall median.")
if property_type_effect < 0.01 and bhk_effect < 0.01:
    print("- Property type and BHK also show negligible rank effects in this dataset.")
print("- Uniform-looking marginals and weak segment relationships are consistent with synthetic or simplified data; this is an inference, not proof.")
print("- These listing records do not support conclusions about actual returns, crime impact, or resale value.")


## 7. EDA findings and limitations

The data-driven summary and charts should be reported together. If the computed effects are negligible and distributions are close to uniform, this file is not capturing realistic market structure. Do not create investment conclusions from visual noise.

- **Investment returns:** not observed; the label is a formula-generated proxy.
- **Crime impact:** not measurable; no crime-rate field is supplied. Security is not a substitute.
- **Infrastructure and resale value:** not measurable; neither infrastructure scores nor resale outcomes are supplied.
- **Asking price:** not transaction price; property mix may differ across cities.
- **Time trends:** the source has no listing or transaction date, so locality/city price trends over time cannot be estimated; v50 supplies cross-sectional rankings instead.

For decision-grade analysis, collect verified transaction and resale history, time-stamped market records, reliable locality identifiers, sourced crime and infrastructure measures, and enough history for time-based validation.


## 8. Export cleaned data and analysis summaries


In [ ]:
cleaned_path = PROCESSED_DIR / "india_housing_prices_cleaned.csv"
df.to_csv(cleaned_path, index=False)
schema.to_csv(REPORT_DIR / "source_data_profile.csv", index=False)
city_summary.to_csv(REPORT_DIR / "city_price_summary.csv", index=False)
area_summary.to_csv(REPORT_DIR / "area_band_summary.csv", index=False)
segment_summary.to_csv(REPORT_DIR / "property_type_bhk_summary.csv", index=False)
amenity_association.to_csv(REPORT_DIR / "amenity_associations.csv", index=False)
diagnostics.to_csv(REPORT_DIR / "eda_diagnostics.csv", index=False)
print(f"Cleaned data: {cleaned_path}")
print(f"Summary tables: {REPORT_DIR}")

# v50 supplemental EDA outputs
distribution_summary.to_csv(REPORT_DIR / "distribution_outlier_summary.csv", index=False)
locality_age_summary.to_csv(REPORT_DIR / "locality_property_age_summary.csv", index=False)
bhk_city_mix.to_csv(REPORT_DIR / "city_bhk_mix_pct.csv")
bhk_city_counts.to_csv(REPORT_DIR / "city_bhk_listing_counts.csv", index=False)
locality_price_summary.to_csv(REPORT_DIR / "locality_price_summary.csv", index=False)
top5_localities.to_csv(REPORT_DIR / "top5_locality_price_ranking.csv", index=False)
facing_summary.to_csv(REPORT_DIR / "facing_price_summary.csv", index=False)
parking_summary.to_csv(REPORT_DIR / "parking_price_summary.csv", index=False)
amenity_summary.to_csv(REPORT_DIR / "amenity_tag_price_summary.csv", index=False)
amenity_count_summary.to_csv(REPORT_DIR / "amenity_count_price_summary.csv", index=False)
transport_summary.to_csv(REPORT_DIR / "transport_accessibility_price_summary.csv", index=False)


## 9. Next steps

1. Review the data-quality tables before modeling.
2. Replace proxy targets with observed outcomes when transaction data becomes available.
3. Fit imputation, encoding, and scaling steps on training data only.
4. Validate with a time-based holdout and compare with simple baselines.
5. Revisit assumptions before using results to guide an investment decision.

See "02_model_selection_and_evaluation.ipynb" for cross-validated model selection and held-out city evaluation.


In [ ]:
# ============================================================
# EXPORT ALL EXECUTIVE REPORT CHARTS
# ============================================================

from pathlib import Path

EXPORT_DIR = PROJECT_ROOT / "reports" / "figures"
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

for fig, filename in EXECUTIVE_FIGURES:
    # High-resolution PNG for reports and presentations
    fig.savefig(
        EXPORT_DIR / f"{filename}.png",
        dpi=300,
        bbox_inches="tight",
        facecolor="white"
    )

    # Vector PDF for high-quality resizing and printing
    fig.savefig(
        EXPORT_DIR / f"{filename}.pdf",
        bbox_inches="tight",
        facecolor="white"
    )

    print(f"Exported: {filename}")

print(f"\nSuccessfully exported {len(EXECUTIVE_FIGURES)} registered EDA and executive charts.")
print(f"Export location: {EXPORT_DIR.resolve()}")